### Loading data in hierarchal HDF5 format

Subject -> Stimulus Modality -> Stimulus Font -> Parity or Control

In [1]:
import os
import pandas as pd
import mne
from pr_fe import FeatureExtractor
import h5py
import numpy as np

In [2]:
input_list_path = 'data/mat_files_cleaned.txt'
data_dir = 'data'
output_base_dir = 'h5_sep'

fe = FeatureExtractor()

In [3]:
def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')
    
    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')
    
    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1

    category = parts[idx] if len(parts) > idx else None
    font_type = parts[idx+1] if len(parts) > idx+1 else None
    condition_code = parts[idx+3] if len(parts) > idx+3 else None
    
    is_20f = font_type == '20F' and len(parts) > idx+2 and parts[idx+2] in ['A', 'S']
    specific_font = parts[idx+2] if is_20f else None
    
    return subject, category, font_type, specific_font, condition_code

In [4]:
def load_and_prepare_csv(filepath):
    df = pd.read_csv(filepath)
    
    df_cond0 = df[df['condition'] == 0].pivot(index='time', columns='channel', values='value')
    df_cond1 = df[df['condition'] == 1].pivot(index='time', columns='channel', values='value')
    
    df_cond0.columns = df_cond0.columns.astype(str)
    df_cond1.columns = df_cond1.columns.astype(str)
    common_channels = sorted(list(set(df_cond0.columns) & set(df_cond1.columns)))
    df_cond0 = df_cond0[common_channels]
    df_cond1 = df_cond1[common_channels]
    
    return df_cond0, df_cond1, common_channels

In [5]:
def process_condition(df_signal, channels, condition_label):
    ch_types = ['eeg'] * len(channels)
    sfreq = fe.sampling_rate
    data = df_signal[channels].T.values 

    info = mne.create_info(ch_names=channels, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(data, info)

    features_df = fe.merging_feature_data(raw, df_signal)
    
    features_df = features_df.apply(pd.to_numeric, errors='coerce')
    features_df = features_df.select_dtypes(include=[np.number])
    features_df = features_df.dropna(axis=1, how='all')
    
    return features_df

In [ ]:
with open(input_list_path, 'r') as f:
    files = [line.strip() for line in f if line.strip()]

for file_rel_path in files:
    csv_rel_path = file_rel_path.replace('.mat', '.csv')
    file_path = os.path.join(data_dir, csv_rel_path)

    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}, skipping.")
        continue

    print(f"Processing {file_path}...")

    subject, category, font_type, specific_font, condition_code = parse_filename(file_rel_path)
    
    if None in [subject, category, font_type]:
        print(f"Skipping {file_path} - missing essential info")
        continue
        
    condition_types = []
    if condition_code and 'Par' in condition_code:
        condition_types.append('Parity')
    if condition_code and 'C1' in condition_code:
        condition_types.append('Control')
    if not condition_types:  
        condition_types = ['Parity', 'Control']
        print(f"Processing both conditions for {file_path}")
    
    df_cond0, df_cond1, common_channels = load_and_prepare_csv(file_path)

    dir_parts = [output_base_dir, f'S{subject}', category, font_type]
    if specific_font:
        dir_parts.append(specific_font)
    
    output_dir = os.path.join(*dir_parts)
    os.makedirs(output_dir, exist_ok=True)
    
    for condition_type in condition_types:
        for cond_num, df in [('0', df_cond0), ('1', df_cond1)]:
            try:
                features_df = process_condition(df, common_channels, cond_num)
                
                output_file = os.path.join(output_dir, f'{condition_type}_{cond_num}.h5')
                group_name = '/'.join(dir_parts[1:])  

                with h5py.File(output_file, 'a') as hdf5_file:
                    group = hdf5_file.require_group(group_name)
                    
                    dataset_name = f'{condition_type}_{cond_num}'
                    if dataset_name in group:
                        print(f"Dataset {dataset_name} exists, skipping.")
                        continue

                    group.create_dataset(dataset_name, 
                                      data=features_df.to_numpy(), 
                                      chunks=True)
                    group.attrs['columns'] = np.array(features_df.columns, dtype='S')
                    group.attrs['condition_type'] = condition_type
                    group.attrs['condition_num'] = cond_num

                print(f"Saved {condition_type}_{cond_num} to {output_file}")
                
            except Exception as e:
                print(f"Failed to process {condition_type}_{cond_num}: {str(e)}")

### Loading data in HDF5 including bins

In [ ]:
import os
import pandas as pd
import numpy as np
import h5py
import mne
from pr_fe import FeatureExtractor

csv_dir = 'csv_data'
output_h5 = 'h5_freq/features_per_bin.h5'
os.makedirs(os.path.dirname(output_h5), exist_ok=True)

fe = FeatureExtractor(sampling_rate=512)

def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')
    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')
    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1
    category = parts[idx] if len(parts) > idx else None
    font_type = parts[idx+1] if len(parts) > idx+1 else None
    condition_code = parts[idx+3] if len(parts) > idx+3 else None
    is_20f = font_type == '20F' and len(parts) > idx+2 and parts[idx+2] in ['A', 'S']
    specific_font = parts[idx+2] if is_20f else None
    return subject, category, font_type, specific_font, condition_code

csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

with h5py.File(output_h5, 'a') as h5f:
    for csv_file in csv_files:
        csv_path = os.path.join(csv_dir, csv_file)
        print(f"Processing {csv_path}...")

        df = pd.read_csv(csv_path)
        subject, category, font_type, specific_font, condition_code = parse_filename(csv_file)

        if 'sequence' not in df.columns:
            df['sequence'] = 0

        grouped = df.groupby(['sequence', 'bin', 'label'])

        for (seq, bin_idx, label), group in grouped:
            df_bin = group.pivot(index='time_in_bin', columns='channel', values='value')
            df_bin = df_bin[sorted(df_bin.columns)]

            ch_types = ['eeg'] * df_bin.shape[1]
            info = mne.create_info(ch_names=df_bin.columns.tolist(), sfreq=fe.sampling_rate, ch_types=ch_types)
            raw = mne.io.RawArray(df_bin.T.values, info)

            freq_features = fe.compute_frequency_features(raw)
            glcm_features = fe.extract_glcm_features(df_bin)
            features_df = pd.merge(freq_features, glcm_features, on='Electrode', how='inner')
            features_array = features_df.drop(columns=['Electrode']).to_numpy()

            group_path = f"S{subject}/{category}/{font_type}"
            if specific_font:
                group_path += f"/{specific_font}"
            condition_group = 'Par' if label in ['odd', 'even'] else 'Control'
            group_path += f"/{condition_group}/sequence_{seq}"
            h5_group = h5f.require_group(group_path)

            ds_name = f"bin{bin_idx}_{label}"
            if ds_name in h5_group:
                continue
            dset = h5_group.create_dataset(ds_name, data=features_array)
            dset.attrs['subject'] = subject
            dset.attrs['category'] = category
            dset.attrs['font_type'] = font_type
            if specific_font:
                dset.attrs['specific_font'] = specific_font
            dset.attrs['condition_code'] = condition_code
            dset.attrs['sequence'] = seq
            dset.attrs['bin'] = bin_idx
            dset.attrs['label'] = label

        print(f"Finished {csv_file}")

print("Hierarchical per-bin feature extraction complete!")

### Including bins and including control

In [1]:
import os
import pandas as pd
import numpy as np
import h5py
import mne
from pr_fe import FeatureExtractor

csv_dir = 'data'
output_h5 = 'h5_freq/features_per_bin.h5'
os.makedirs(os.path.dirname(output_h5), exist_ok=True)

fe = FeatureExtractor(sampling_rate=512)

def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')
    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')
    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1
    category = parts[idx] if len(parts) > idx else None
    font_type = parts[idx+1] if len(parts) > idx+1 else None
    condition_code = parts[idx+3] if len(parts) > idx+3 else None
    is_20f = font_type == '20F' and len(parts) > idx+2 and parts[idx+2] in ['A', 'S']
    specific_font = parts[idx+2] if is_20f else None
    return subject, category, font_type, specific_font, condition_code

csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

with h5py.File(output_h5, 'a') as h5f:
    for csv_file in csv_files:
        csv_path = os.path.join(csv_dir, csv_file)
        print(f"Processing {csv_path}...")

        df = pd.read_csv(csv_path)
        subject, category, font_type, specific_font, condition_code = parse_filename(csv_file)

        if 'sequence' not in df.columns:
            df['sequence'] = 0

        grouped = df.groupby(['sequence', 'bin', 'condition', 'odd_even'])

        for (seq, bin_idx, condition, odd_even), group in grouped:
            df_bin = group.pivot(index='time_in_bin', columns='channel', values='value')
            df_bin = df_bin[sorted(df_bin.columns)]

            ch_types = ['eeg'] * df_bin.shape[1]
            info = mne.create_info(ch_names=df_bin.columns.tolist(), sfreq=fe.sampling_rate, ch_types=ch_types)
            raw = mne.io.RawArray(df_bin.T.values, info)

            freq_features = fe.compute_frequency_features(raw)
            glcm_features = fe.extract_glcm_features(df_bin)
            features_df = pd.merge(freq_features, glcm_features, on='Electrode', how='inner')
            features_array = features_df.drop(columns=['Electrode']).to_numpy()

            group_path = f"S{subject}/{category}/{font_type}"
            if specific_font:
                group_path += f"/{specific_font}"
            condition_group = 'Par' if condition == 'parity' else 'Control'
            group_path += f"/{condition_group}/sequence_{seq}"
            h5_group = h5f.require_group(group_path)

            ds_name = f"bin{bin_idx}_{odd_even}"
            if ds_name in h5_group:
                continue
            dset = h5_group.create_dataset(ds_name, data=features_array)
            dset.attrs['subject'] = subject
            dset.attrs['category'] = category
            dset.attrs['font_type'] = font_type
            if specific_font:
                dset.attrs['specific_font'] = specific_font
            dset.attrs['condition_code'] = condition_code
            dset.attrs['sequence'] = seq
            dset.attrs['bin'] = bin_idx
            dset.attrs['condition'] = condition
            dset.attrs['odd_even'] = odd_even

        print(f"Finished {csv_file}")

print("Hierarchical per-bin feature extraction complete!")

Processing data/epbin_Dig_20F_A_Par_13_rr_fixFC6_fixF4_ica_ep1_but_chanlocs_chansel_chanlabels_S10.csv...
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.13

/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid va

Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)


/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid va

Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)


...

Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)
Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)
Creating RawArray with float64 data, n_channels=70, n_times=69
    Range : 0 ... 68 =      0.000 ...     0.133 secs
Ready.
Effective window size : 0.135 (s)


/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid va

Creating RawArray with float64 data, n_channels=70, n_times=68
    Range : 0 ... 67 =      0.000 ...     0.131 secs
Ready.
Effective window size : 0.133 (s)


/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/a1234/miniforge3/envs/env/lib/python3.13/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


KeyboardInterrupt: 

### Unlabeled data

In [ ]:
import os
import pandas as pd
import numpy as np
import h5py
import mne
from pr_fe import FeatureExtractor

csv_dir = 'unlabeled_data'  # your unlabeled CSVs
output_h5 = 'h5_freq/features_windowed_unlabeled_experimental.h5'
os.makedirs(os.path.dirname(output_h5), exist_ok=True)

fe = FeatureExtractor(sampling_rate=512)

WINDOW_SIZE = 128       # number of samples per window
WINDOW_STEP = 64        # step size (overlap if < window size)

def parse_filename(filename):
    base = os.path.basename(filename).replace('.csv', '')
    parts = base.split('_')
    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')
    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1
    category = parts[idx] if len(parts) > idx else None
    font_type = parts[idx+1] if len(parts) > idx+1 else None
    condition_code = parts[idx+3] if len(parts) > idx+3 else None
    is_20f = font_type == '20F' and len(parts) > idx+2 and parts[idx+2] in ['A', 'S']
    specific_font = parts[idx+2] if is_20f else None
    return subject, category, font_type, specific_font, condition_code

csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

with h5py.File(output_h5, 'a') as h5f:
    for csv_file in csv_files:
        csv_path = os.path.join(csv_dir, csv_file)
        print(f"Processing {csv_path}...")

        df = pd.read_csv(csv_path)
        subject, category, font_type, specific_font, condition_code = parse_filename(csv_file)

        # Determine Parity vs Control from filename
        parity_control_label = 'Par' if 'Par' in csv_file else 'Control'

        # Loop over sequences (CSV column 'condition' = sequence 0 or 1)
        for seq in df['condition'].unique():
            df_seq = df[df['condition'] == seq]

            # Pivot channels x time
            pivot_df = df_seq.pivot(index='time', columns='channel', values='value')
            pivot_df = pivot_df[sorted(pivot_df.columns)]
            pivot_df.columns = pivot_df.columns.astype(str)
            
            ch_types = ['eeg'] * pivot_df.shape[1]
            info = mne.create_info(ch_names=pivot_df.columns.tolist(), sfreq=fe.sampling_rate, ch_types=ch_types)

            n_samples = pivot_df.shape[0]
            window_starts = np.arange(0, n_samples - WINDOW_SIZE + 1, WINDOW_STEP)

            # Build HDF5 group path per sequence under Par/Control
            group_path = f"S{subject}/{category}/{font_type}"
            if specific_font:
                group_path += f"/{specific_font}"
            group_path += f"/{parity_control_label}/sequence_{seq}"
            h5_group = h5f.require_group(group_path)

            # Save windows inside each sequence
            for i, start in enumerate(window_starts):
                end = start + WINDOW_SIZE
                window_df = pivot_df.iloc[start:end]

                raw_window = mne.io.RawArray(window_df.T.values, info)
                freq_features = fe.compute_frequency_features(raw_window)
                glcm_features = fe.extract_glcm_features(window_df)
                ssvep_df = fe.compute_ssvep_feature(raw_window)
                features_df = pd.merge(freq_features, glcm_features, on='Electrode', how='inner')
                features_df = pd.merge(features_df, ssvep_df, on='Electrode', how='inner')
                features_array = features_df.drop(columns=['Electrode']).to_numpy()

                dset_name = f"window_{i}"
                if dset_name in h5_group:
                    continue
                dset = h5_group.create_dataset(dset_name, data=features_array)

                # Metadata
                dset.attrs['subject'] = subject
                dset.attrs['category'] = category
                dset.attrs['font_type'] = font_type
                if specific_font:
                    dset.attrs['specific_font'] = specific_font
                dset.attrs['condition'] = parity_control_label  # Parity or Control
                dset.attrs['sequence'] = seq                  # 0 or 1
                dset.attrs['window_start'] = start
                dset.attrs['window_end'] = end

        print(f"Finished {csv_file}")

print("Hierarchical windowed feature extraction complete!")

Processing unlabeled_data\epbin_Dig_1F_C1_21_rr_fixAF7_fixF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S19.csv...


Matplotlib is building the font cache; this may take a moment.


KeyboardInterrupt: 